# Notebook 4a: Baseline 1, feature moment matching (as in Xiang et al., 2026)

**This is a rebuild.** The first version of this notebook adjusted the *field* features and reused the old classifier.
The paper does it the other way round, so this version follows the paper (Section 2.4).

### The method, as the paper describes it
1. Take the **PlantVillage training features** (the 2048 numbers per image from the trained ResNet-50).
2. **Reshape them** so that each of the 2048 features has the **mean and spread of the field images** (estimated from the
   Setting B calibration images, used **without labels**).
3. **Train a new linear classifier** on these reshaped source features, using the source labels.
4. Apply that new classifier to the **raw field features** and measure.

In words: instead of moving the field images toward the lab, we make the lab features *look like the field's statistics*
and learn a classifier that is comfortable with that. The classifier is **new**, not the one fine-tuned in notebook 02.

### What the paper does not say (our choices)
* How the linear classifier is trained: we use ordinary **logistic regression** (a linear classifier with L2 penalty).
* Where the source statistics come from: the paper says "source-domain channel statistics"; we compute them from the
  PlantVillage **training** features.

### Why a control is included
Because this method trains a *new* classifier, part of any change might come from re-training the classifier and not
from the alignment. So we also train the same classifier on **un-reshaped** source features. Comparing the two tells
us how much the alignment itself contributes.

### Plan
1. Settings
2. Load what we need
3. The alignment, on a tiny example
4. One split in detail
5. The control: same classifier, no alignment
6. All 10 splits, then save
7. Evaluate with notebook 3b

## Step 1: Settings

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

In [2]:
# ---- Edit these -------------------------------------------------------------
METHOD = "moment_matching"        # name of this method (used in the saved file names)
CONTROL_METHOD = "linear_head"    # name of the control (new classifier, no alignment)
SOURCE_METHOD = "source_only"     # whose saved outputs we start from
TRAIN_SEED = 42

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
SEEDS = list(range(42, 52))       # the 10 Setting B splits

EPS = 1e-6                        # tiny number added before dividing, to avoid dividing by zero
MAX_ITER = 200                    # how long the logistic regression may optimise (raise it if it warns about convergence)
C = 1.0                           # strength of the L2 penalty (sklearn default; larger = weaker penalty)
# -----------------------------------------------------------------------------

## Step 2: Load what we need

* From notebook 02: the field features (`pd_known`, `pd_unknown`), the labels and image paths, and the lab logits.
* From notebook 02b: the **PlantVillage training features** and their labels.

In [3]:
source = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(source["classes"])

field_features = {g: source[f"{g}_features"].astype(np.float32) for g in ["pd_known", "pd_unknown"]}
field_labels = source["pd_known_labels"]
field_paths = source["pd_known_paths"]

train = np.load(OUT_DIR / f"features_pv_train_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
train_features = train["features"].astype(np.float32)
train_labels = train["labels"]

print("source training features:", train_features.shape)
print("field features (known):  ", field_features["pd_known"].shape)
print("field features (unknown):", field_features["pd_unknown"].shape)
assert train_features.shape[1] == field_features["pd_known"].shape[1], "Feature sizes differ"
assert set(train_labels) == set(range(len(classes))), "Not every class appears in the training features"

source training features: (18601, 2048)
field features (known):   (2018, 2048)
field features (unknown): (466, 2048)


## Step 3: The alignment, on a tiny example

For **one** feature, suppose:
* source (lab) images: average 2, spread 1
* target (field) images: average 5, spread 3

A source image has the value 3 for this feature. To give it the *target's* statistics: subtract the source average,
divide by the source spread, multiply by the target spread, add the target average:

(3 - 2) / 1 x 3 + 5 = **8**

"One spread above the average" stays "one spread above the average", but now measured in the field's units. We do this
for all 2048 features at once.

In [4]:
def moment_match(x, from_mean, from_std, to_mean, to_std):
    # Reshape x so that data with the 'from' mean/std ends up with the 'to' mean/std
    return (x - from_mean) / (from_std + EPS) * to_std + to_mean


print("Toy example:", moment_match(3.0, from_mean=2.0, from_std=1.0, to_mean=5.0, to_std=3.0), "(by hand: 8)")

Toy example: 7.999997000003001 (by hand: 8)


## Step 4: One split in detail

First we link the field images to the manifest to know which are calibration and which are test in each split (as in
notebooks 03 and 3b).

In [5]:
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]

known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")
split_info = known_rows.loc[field_paths, split_columns].reset_index(drop=True)

# Source statistics: computed once, from the PlantVillage TRAINING features
source_mean = train_features.mean(axis=0)
source_std = train_features.std(axis=0)
print("Source statistics from", len(train_features), "training images")

Source statistics from 18601 training images


In [6]:
seed = 42
is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
is_test = ~is_cal

# Target statistics: from this split's calibration images only (labels NOT used)
target_mean = field_features["pd_known"][is_cal].mean(axis=0)
target_std = field_features["pd_known"][is_cal].std(axis=0)
print(f"Target statistics from {is_cal.sum()} calibration images")

# Reshape ALL source training features to the target's statistics
aligned_train = moment_match(train_features, source_mean, source_std, target_mean, target_std)

print("Average feature value:  source", round(float(train_features.mean()), 4),
      "| aligned source", round(float(aligned_train.mean()), 4),
      "| field", round(float(field_features['pd_known'][is_cal].mean()), 4))

Target statistics from 202 calibration images
Average feature value:  source 0.547 | aligned source 0.5125 | field 0.5125


Now train a **new linear classifier** on the aligned source features. A linear classifier is one weight matrix (like the
final layer of the network). `LogisticRegression` learns it from the labelled source images. (This can take a minute.)

In [7]:
start = time.time()
classifier = LogisticRegression(max_iter=MAX_ITER, C=C)
classifier.fit(aligned_train, train_labels)
print(f"Trained in {time.time() - start:.0f} seconds ({classifier.n_iter_[0]} iterations)")

assert list(classifier.classes_) == list(range(len(classes))), "Unexpected class order"

Trained in 2 seconds (13 iterations)


In [8]:
# Apply the new classifier to the RAW field features
new_logits = classifier.decision_function(field_features["pd_known"])
old_logits = source["pd_known_logits"]            # the original fine-tuned network's scores

def accuracy(logits, mask):
    return float((logits[mask].argmax(axis=1) == field_labels[mask]).mean())

print(f"Accuracy on the {is_test.sum()} test images (labels used only for scoring):")
print(f"  original network (source-only): {accuracy(old_logits, is_test):.4f}")
print(f"  moment matching (new classifier): {accuracy(new_logits, is_test):.4f}")

Accuracy on the 1816 test images (labels used only for scoring):
  original network (source-only): 0.2390
  moment matching (new classifier): 0.2841


**Questions to think about:**
* Did accuracy change? The paper reports roughly 0.32 for the original model and about 0.37 for moment matching.
* The classifier was trained **only on lab images** (reshaped). What does it never get to see?

## Step 5: The control, same classifier but no alignment

We train the **same kind of new classifier** on the *original, un-reshaped* source features. If this scores about the
same as moment matching, the gain comes from re-training the classifier and not from the alignment. If moment matching
is clearly better, the alignment is doing real work.

This control does not depend on any split (it uses no target images), so we train it once.

In [9]:
start = time.time()
control = LogisticRegression(max_iter=MAX_ITER, C=C)
control.fit(train_features, train_labels)
print(f"Trained in {time.time() - start:.0f} seconds")

control_logits = {g: control.decision_function(field_features[g]) for g in ["pd_known", "pd_unknown"]}
print(f"Control accuracy on the split-{seed} test images: {accuracy(control_logits['pd_known'], is_test):.4f}")

Trained in 2 seconds
Control accuracy on the split-42 test images: 0.2285


## Step 6: All 10 splits, then save

For each split we recompute the target statistics from that split's calibration images, reshape the source features,
train a new classifier, and score **all** field images (known and unknown) with the raw field features. We keep one
set of logits per split.

The lab (PlantVillage) logits are copied unchanged from notebook 02, as the lab reference. (Training all 10 classifiers
takes several minutes on a CPU.)

In [10]:
save_dict = {"classes": np.array(classes)}
for key in ["epochs_run", "debug_mode"]:                 # carry over how the source model was trained
    if key in source.files:
        save_dict[key] = source[key]

for g in ["pv_val", "pv_test"]:
    save_dict[f"{g}_logits"] = source[f"{g}_logits"]
for g in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    save_dict[f"{g}_labels"] = source[f"{g}_labels"]
    save_dict[f"{g}_paths"] = source[f"{g}_paths"]

for seed in SEEDS:
    start = time.time()
    is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
    t_mean = field_features["pd_known"][is_cal].mean(axis=0)
    t_std = field_features["pd_known"][is_cal].std(axis=0)

    clf = LogisticRegression(max_iter=MAX_ITER, C=C)
    clf.fit(moment_match(train_features, source_mean, source_std, t_mean, t_std), train_labels)

    for g in ["pd_known", "pd_unknown"]:
        save_dict[f"{g}_logits_B_seed{seed}"] = clf.decision_function(field_features[g]).astype(np.float32)
    print(f"split {seed}: done in {time.time() - start:.0f} s")

output_file = OUT_DIR / f"outputs_{METHOD}_seed{TRAIN_SEED}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file)

split 42: done in 3 s
split 43: done in 3 s
split 44: done in 2 s
split 45: done in 2 s
split 46: done in 3 s
split 47: done in 2 s
split 48: done in 3 s
split 49: done in 3 s
split 50: done in 4 s
split 51: done in 4 s
Saved outputs\outputs_moment_matching_seed42.npz


In [11]:
# The control: one set of logits shared by all splits (same format as source-only)
control_dict = {k: v for k, v in save_dict.items() if "_logits_B_seed" not in k}
for g in ["pd_known", "pd_unknown"]:
    control_dict[f"{g}_logits"] = control_logits[g].astype(np.float32)

control_file = OUT_DIR / f"outputs_{CONTROL_METHOD}_seed{TRAIN_SEED}.npz"
np.savez_compressed(control_file, **control_dict)
print("Saved", control_file)

Saved outputs\outputs_linear_head_seed42.npz


## Step 7: Evaluate with notebook 3b

Open notebook 3b and set
```python
METHODS = ["source_only", "linear_head", "moment_matching"]
```
then run it. You will get one row for each of the three, scored on the same 10 splits with the same seven metrics.

How to read the three rows together:
* `source_only` vs `linear_head`: what does simply *re-training the classifier* on frozen features do?
* `linear_head` vs `moment_matching`: how much does the *alignment* add on top?
* `moment_matching` vs the paper: accuracy about 0.366, calibrated ECE about 0.310, Brier about 0.925.

### Next
**Notebook 4b: AdaBN** (needs PyTorch and the images again), then **4c: DANN**, which also uses the source training
features we just extracted.